# Importing the necessary libraries

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import optuna
import os

# Loading the datasets

In [22]:
blr_df = pd.read_csv('../Data/Processed/blr_df.csv')
hyd_df = pd.read_csv('../Data/Processed/hyd_df.csv')
pune_df = pd.read_csv('../Data/Processed/pune_df.csv')

In [23]:
blr_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,12.498648,916.589625,0.608896,327.505071,93.194473,30.939528
1,2003-01-02,12.825784,918.004624,3.006925,290.310937,93.578218,31.179113
2,2003-01-03,12.970981,918.453619,2.872020,279.921010,94.227231,33.045705
3,2003-01-04,12.445614,917.966755,2.678422,274.952584,93.297114,38.266933
4,2003-01-05,14.071074,918.579933,3.104962,275.257170,94.353249,33.185394
...,...,...,...,...,...,...,...
6553,2020-12-25,13.500707,916.151180,2.152634,261.457477,95.470006,32.118758
6554,2020-12-26,12.901577,917.194971,2.312996,253.686322,95.361429,32.068009
6555,2020-12-27,12.950693,916.302854,2.260313,252.095788,94.854902,34.539327
6556,2020-12-28,10.830827,916.237697,2.003774,254.229795,93.468912,31.310399


In [24]:
pune_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,7.100958,941.961323,0.608896,327.505071,91.114710,34.246204
1,2003-01-02,10.484796,942.675433,3.006925,290.310937,91.934631,35.734294
2,2003-01-03,12.606104,942.881457,2.872020,279.921010,92.393007,32.590960
3,2003-01-04,12.646417,943.009227,2.678422,274.952584,92.312594,35.435619
4,2003-01-05,13.066706,943.762527,3.104962,275.257170,92.569458,34.235347
...,...,...,...,...,...,...,...
6553,2020-12-25,12.096190,941.053885,2.152634,261.457477,92.849653,33.209944
6554,2020-12-26,13.381471,942.263093,2.312996,253.686322,93.400294,31.352555
6555,2020-12-27,13.819016,941.287692,2.260313,252.095788,93.560516,33.862652
6556,2020-12-28,13.831078,940.630939,2.003774,254.229795,93.895661,32.318171


In [25]:
hyd_df

,Date,DPT,AP,WS,WSD,RH,LST
0,2003-01-01,9.801921,952.321702,2.702138,228.989644,91.137446,35.253027
1,2003-01-02,12.894156,953.896739,2.974241,287.669929,92.938629,31.660535
2,2003-01-03,16.945643,954.471306,3.015529,308.536731,96.690377,30.085849
3,2003-01-04,17.761199,954.172469,1.865083,299.395258,98.016959,30.569864
4,2003-01-05,14.281283,954.395644,2.287668,265.010524,93.245268,31.878895
...,...,...,...,...,...,...,...
6553,2020-12-25,13.330685,951.670802,2.015309,292.172202,94.673562,30.396700
6554,2020-12-26,14.027632,952.521458,1.736367,284.297801,94.788311,32.262344
6555,2020-12-27,13.223768,951.342613,1.754041,288.307250,94.039085,33.796627
6556,2020-12-28,12.504071,951.101661,2.012608,299.893708,93.248170,31.748070


# Splitting the Input and Predictor Variables

In [26]:
features = ['AP', 'DPT', 'WS', 'WSD', 'RH']
target = 'LST'

In [27]:
blr_X = blr_df[features].values
blr_y = blr_df[target].values

In [28]:
hyd_X = hyd_df[features].values
hyd_y = hyd_df[target].values

In [29]:
pune_X = pune_df[features].values
pune_y = pune_df[target].values

# Scaling the data

In [30]:
scaler = StandardScaler()

In [31]:
blr_X_scaled = scaler.fit_transform(blr_X)
hyd_X_scaled = scaler.transform(hyd_X)
pune_X_scaled = scaler.transform(pune_X)

# Splitting it into Training data and Testing Data

In [32]:
train_size = int(0.8 * len(blr_X_scaled))
blr_X_train = blr_X_scaled[:train_size]
blr_y_train = blr_y[:train_size]
blr_X_test = blr_X_scaled[train_size:]
blr_y_test = blr_y[train_size:]

In [33]:
train_size = int(0.8 * len(hyd_X_scaled))
hyd_X_train = hyd_X_scaled[:train_size]
hyd_y_train = hyd_y[:train_size]
hyd_X_test = hyd_X_scaled[train_size:]
hyd_y_test = hyd_y[train_size:]

In [34]:
train_size = int(0.8 * len(pune_X_scaled))
pune_X_train = pune_X_scaled[:train_size]
pune_y_train = pune_y[:train_size]
pune_X_test = pune_X_scaled[train_size:]
pune_y_test = pune_y[train_size:]

# Converting it into Pytorch Tensor for Further Analysis

In [35]:
blr_X_train_tensor  = torch.tensor(blr_X_train, dtype=torch.float32)
blr_y_train_tensor  = torch.tensor(blr_y_train, dtype=torch.float32).unsqueeze(1)
blr_X_test_tensor   = torch.tensor(blr_X_test, dtype=torch.float32)
blr_y_test_tensor   = torch.tensor(blr_y_test, dtype=torch.float32).unsqueeze(1)

hyd_X_train_tensor  = torch.tensor(hyd_X_train, dtype=torch.float32)
hyd_y_train_tensor  = torch.tensor(hyd_y_train, dtype=torch.float32).unsqueeze(1)
hyd_X_test_tensor   = torch.tensor(hyd_X_test, dtype=torch.float32)
hyd_y_test_tensor   = torch.tensor(hyd_y_test, dtype=torch.float32).unsqueeze(1)

pune_X_train_tensor = torch.tensor(pune_X_train, dtype=torch.float32)
pune_y_train_tensor = torch.tensor(pune_y_train, dtype=torch.float32).unsqueeze(1)
pune_X_test_tensor  = torch.tensor(pune_X_test, dtype=torch.float32)
pune_y_test_tensor  = torch.tensor(pune_y_test, dtype=torch.float32).unsqueeze(1)

# Defining the ANN Model

In [36]:
class ANN(nn.Module):
    def __init__(self, input_size, hidden_sizes, dropout_rate):
        super(ANN, self).__init__()
        layers = []
        in_features = input_size

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(in_features, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            in_features = hidden_size

        layers.append(nn.Linear(in_features, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

# Training the Model with the Hyperparameter Tuning done using Optuna

In [37]:
def objective(trial, X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor):
    hidden_layer_count = trial.suggest_int("n_layers", 1, 2)
    hidden_sizes = [trial.suggest_int(f"n_units_l{i}", 50, 500) for i in range(hidden_layer_count)]
    dropout = trial.suggest_float("dropout", 0.0, 0.4)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8])
    epochs = trial.suggest_int("epochs", 50, 150)

    model = ANN(input_size=X_train.shape[1], hidden_sizes=hidden_sizes, dropout_rate=dropout)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
    loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

    model.train()
    for epoch in range(epochs):
        for batch_X, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluate on test data
    model.eval()
    with torch.no_grad():
        predictions = model(X_test_tensor)
        mse = criterion(predictions, y_test_tensor).item()
    return mse

In [38]:
cities = {
    'Bangalore': (blr_X_train_tensor, blr_y_train_tensor, blr_X_test_tensor, blr_y_test_tensor),
    'Hyderabad': (hyd_X_train_tensor, hyd_y_train_tensor, hyd_X_test_tensor, hyd_y_test_tensor),
    'Pune': (pune_X_train_tensor, pune_y_train_tensor, pune_X_test_tensor, pune_y_test_tensor),
}

In [39]:
results = {}

In [40]:
for city, (X_train, y_train, X_test, y_test) in cities.items():
    print(f"Running Optuna for {city}...")
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, X_train, y_train, X_test, y_test), n_trials=30)
    
    best_trial = study.best_trial
    print(f"{city} Best MSE: {best_trial.value:.4f}")
    print(f"Best Parameters: {best_trial.params}\n")

    results[city] = best_trial

[I 2025-04-22 11:21:36,556] A new study created in memory with name: no-name-387752b2-73f9-4027-b591-4a4a4b8a82a2


Running Optuna for Bangalore...


[I 2025-04-22 11:21:52,549] Trial 0 finished with value: 7.570242404937744 and parameters: {'n_layers': 1, 'n_units_l0': 71, 'dropout': 0.12597039170855867, 'lr': 0.003550179718119552, 'batch_size': 8, 'epochs': 119}. Best is trial 0 with value: 7.570242404937744.
[I 2025-04-22 11:22:12,059] Trial 1 finished with value: 7.332128047943115 and parameters: {'n_layers': 2, 'n_units_l0': 380, 'n_units_l1': 363, 'dropout': 0.21999322098673055, 'lr': 0.000599224461642003, 'batch_size': 8, 'epochs': 60}. Best is trial 1 with value: 7.332128047943115.
[I 2025-04-22 11:22:25,641] Trial 2 finished with value: 7.856171607971191 and parameters: {'n_layers': 1, 'n_units_l0': 93, 'dropout': 0.12781103168376085, 'lr': 0.005193391176239543, 'batch_size': 8, 'epochs': 102}. Best is trial 1 with value: 7.332128047943115.
[I 2025-04-22 11:22:49,745] Trial 3 finished with value: 9.110514640808105 and parameters: {'n_layers': 2, 'n_units_l0': 376, 'n_units_l1': 280, 'dropout': 0.039044139196513686, 'lr': 0.

Bangalore Best MSE: 7.1392
Best Parameters: {'n_layers': 2, 'n_units_l0': 400, 'n_units_l1': 175, 'dropout': 0.17405013132252034, 'lr': 0.0011864757981162946, 'batch_size': 8, 'epochs': 88}

Running Optuna for Hyderabad...


[I 2025-04-22 11:34:42,320] Trial 0 finished with value: 10.2513427734375 and parameters: {'n_layers': 2, 'n_units_l0': 398, 'n_units_l1': 495, 'dropout': 0.15883310067937076, 'lr': 0.004195412551561011, 'batch_size': 8, 'epochs': 100}. Best is trial 0 with value: 10.2513427734375.
[I 2025-04-22 11:35:14,376] Trial 1 finished with value: 10.119962692260742 and parameters: {'n_layers': 2, 'n_units_l0': 333, 'n_units_l1': 390, 'dropout': 0.23390798094074705, 'lr': 0.008005510913968743, 'batch_size': 8, 'epochs': 75}. Best is trial 1 with value: 10.119962692260742.
[I 2025-04-22 11:35:28,301] Trial 2 finished with value: 8.92397403717041 and parameters: {'n_layers': 1, 'n_units_l0': 236, 'dropout': 0.0424096522405379, 'lr': 0.0005845481821681345, 'batch_size': 8, 'epochs': 99}. Best is trial 2 with value: 8.92397403717041.
[I 2025-04-22 11:35:58,480] Trial 3 finished with value: 8.874987602233887 and parameters: {'n_layers': 2, 'n_units_l0': 458, 'n_units_l1': 484, 'dropout': 0.0539812361

Hyderabad Best MSE: 8.8473
Best Parameters: {'n_layers': 2, 'n_units_l0': 435, 'n_units_l1': 273, 'dropout': 0.33602141166386784, 'lr': 0.0001991418216435955, 'batch_size': 8, 'epochs': 116}

Running Optuna for Pune...


[I 2025-04-22 11:50:51,481] Trial 0 finished with value: 11.264348983764648 and parameters: {'n_layers': 1, 'n_units_l0': 447, 'dropout': 0.028526656876352696, 'lr': 0.00035871807879871387, 'batch_size': 8, 'epochs': 138}. Best is trial 0 with value: 11.264348983764648.
[I 2025-04-22 11:51:17,503] Trial 1 finished with value: 13.486594200134277 and parameters: {'n_layers': 2, 'n_units_l0': 278, 'n_units_l1': 300, 'dropout': 0.29439242134404414, 'lr': 0.0005062775696318495, 'batch_size': 8, 'epochs': 60}. Best is trial 0 with value: 11.264348983764648.
[I 2025-04-22 11:51:42,113] Trial 2 finished with value: 11.685722351074219 and parameters: {'n_layers': 1, 'n_units_l0': 421, 'dropout': 0.24806352831023717, 'lr': 0.0008581857555849387, 'batch_size': 8, 'epochs': 95}. Best is trial 0 with value: 11.264348983764648.
[I 2025-04-22 11:52:13,978] Trial 3 finished with value: 14.497031211853027 and parameters: {'n_layers': 2, 'n_units_l0': 422, 'n_units_l1': 252, 'dropout': 0.277632527731798

Pune Best MSE: 10.7955
Best Parameters: {'n_layers': 1, 'n_units_l0': 451, 'dropout': 0.16218964509138323, 'lr': 0.001504634398533723, 'batch_size': 8, 'epochs': 125}



In [41]:
results

{'Bangalore': FrozenTrial(number=5, state=TrialState.COMPLETE, values=[7.139224052429199], datetime_start=datetime.datetime(2025, 4, 22, 11, 23, 9, 534554), datetime_complete=datetime.datetime(2025, 4, 22, 11, 23, 34, 413962), params={'n_layers': 2, 'n_units_l0': 400, 'n_units_l1': 175, 'dropout': 0.17405013132252034, 'lr': 0.0011864757981162946, 'batch_size': 8, 'epochs': 88}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_layers': IntDistribution(high=2, log=False, low=1, step=1), 'n_units_l0': IntDistribution(high=500, log=False, low=50, step=1), 'n_units_l1': IntDistribution(high=500, log=False, low=50, step=1), 'dropout': FloatDistribution(high=0.4, log=False, low=0.0, step=None), 'lr': FloatDistribution(high=0.01, log=True, low=0.0001, step=None), 'batch_size': CategoricalDistribution(choices=(8,)), 'epochs': IntDistribution(high=150, log=False, low=50, step=1)}, trial_id=5, value=None),
 'Hyderabad': FrozenTrial(number=15, state=TrialState.COMPLETE, va

# Evaluating the Model and Visualizing the Results

In [42]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [43]:
def evaluate_predictions(y_true, y_pred):
    y_true = y_true.squeeze()
    y_pred = y_pred.squeeze()
    
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    # NSE
    nse = 1 - np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2)
    
    # RSR = RMSE / STDEV of observed
    rsr = rmse / np.std(y_true)
    
    # PBIAS
    pbias = 100 * np.sum(y_true - y_pred) / np.sum(y_true)

    return {
        "MSE": mse,
        "MAE": mae,
        "R²": r2,
        "NSE": nse,
        "RSR": rsr,
        "PBIAS": pbias
    }

In [44]:
def plot_predictions(y_true, y_pred, title="Prediction vs Ground Truth"):
    plt.figure(figsize=(10, 5))
    plt.plot(y_true.squeeze(), label="True", alpha=0.7)
    plt.plot(y_pred.squeeze(), label="Predicted", alpha=0.7)
    plt.title(title)
    plt.xlabel("Time Step")
    plt.ylabel("LST")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [45]:
from optuna.visualization.matplotlib import plot_optimization_history

def plot_optuna_loss(study, title="Optuna Loss Over Trials"):
    fig = plot_optimization_history(study)
    fig.gca().set_title(title)
    plt.tight_layout()
    plt.show()

In [46]:
def train_final_model(X_train, y_train, X_test, y_test, best_params):
    hidden_sizes = [best_params[f"n_units_l{i}"] for i in range(best_params["n_layers"])]
    model = ANN(X_train.shape[1], hidden_sizes, best_params["dropout"])
    optimizer = torch.optim.Adam(model.parameters(), lr=best_params["lr"])
    criterion = nn.MSELoss()

    dataset = torch.utils.data.TensorDataset(X_train, y_train)
    loader = torch.utils.data.DataLoader(dataset, batch_size=best_params["batch_size"], shuffle=True)

    model.train()
    for epoch in range(best_params["epochs"]):
        for X_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

    return model

In [47]:
import plotly.graph_objects as go
def plot_predictions_plotly(y_true, y_pred, title="Prediction vs Ground Truth (Test Data)"):
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        y=y_true.squeeze(),
        mode='lines',
        name='True',
        line=dict(color='blue')
    ))

    fig.add_trace(go.Scatter(
        y=y_pred.squeeze(),
        mode='lines',
        name='Predicted',
        line=dict(color='orange')
    ))

    fig.update_layout(
        title=title,
        xaxis_title="Time Step (Test Set)",
        yaxis_title="LST",
        legend=dict(x=0.01, y=0.99),
        template='plotly_white',
        height=500,
        width=1000
    )

    fig.show()

In [48]:
blr_best_params = results["Bangalore"].params
pune_best_params = results["Pune"].params
hyd_best_params = results["Hyderabad"].params

In [49]:
blr_model = train_final_model(
    blr_X_train_tensor, blr_y_train_tensor,
    blr_X_test_tensor, blr_y_test_tensor,
    blr_best_params
)

In [50]:
blr_model.eval()

ANN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=400, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.17405013132252034, inplace=False)
    (3): Linear(in_features=400, out_features=175, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.17405013132252034, inplace=False)
    (6): Linear(in_features=175, out_features=1, bias=True)
  )
)

In [51]:
with torch.no_grad():
    y_pred_blr = blr_model(blr_X_test_tensor).numpy()
    y_true_blr = blr_y_test_tensor.numpy()

In [52]:
metrics_blr = evaluate_predictions(y_true_blr, y_pred_blr)
print("Bangalore Metrics:")
for k, v in metrics_blr.items():
    print(f"{k}: {v:.4f}")

Bangalore Metrics:
MSE: 7.9490
MAE: 2.1733
R²: 0.6562
NSE: 0.6562
RSR: 0.5864
PBIAS: -1.7650


In [53]:
plot_predictions_plotly(y_true_blr, y_pred_blr, title="Bangalore's Prediction vs Ground Truth (Test)")

In [54]:
hyd_model = train_final_model(
    hyd_X_train_tensor, hyd_y_train_tensor,
    hyd_X_test_tensor, hyd_y_test_tensor,
    hyd_best_params
)

In [55]:
hyd_model.eval()

ANN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=435, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.33602141166386784, inplace=False)
    (3): Linear(in_features=435, out_features=273, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.33602141166386784, inplace=False)
    (6): Linear(in_features=273, out_features=1, bias=True)
  )
)

In [56]:
with torch.no_grad():
    y_pred_hyd = hyd_model(hyd_X_test_tensor).numpy()
    y_true_hyd = hyd_y_test_tensor.numpy()

In [57]:
metrics_hyd = evaluate_predictions(y_true_hyd, y_pred_hyd)
print("Hyderabad Metrics:")
for k, v in metrics_hyd.items():
    print(f"{k}: {v:.4f}")

Hyderabad Metrics:
MSE: 9.0665
MAE: 2.2921
R²: 0.6035
NSE: 0.6035
RSR: 0.6297
PBIAS: -1.2928


In [58]:
plot_predictions_plotly(y_true_hyd, y_pred_hyd, title="Hyderabad's Prediction vs Ground Truth (Test)")

In [59]:
pune_model = train_final_model(
    pune_X_train_tensor, pune_y_train_tensor,
    pune_X_test_tensor, pune_y_test_tensor,
    pune_best_params
)

In [60]:
pune_model.eval()

ANN(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=451, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.16218964509138323, inplace=False)
    (3): Linear(in_features=451, out_features=1, bias=True)
  )
)

In [61]:
with torch.no_grad():
    y_pred_pune = pune_model(pune_X_test_tensor).numpy()
    y_true_pune = pune_y_test_tensor.numpy()

In [62]:
metrics_pune = evaluate_predictions(y_true_pune, y_pred_pune)
print("Pune's Metrics:")
for k, v in metrics_pune.items():
    print(f"{k}: {v:.4f}")

Pune's Metrics:
MSE: 14.4300
MAE: 2.8466
R²: 0.6809
NSE: 0.6809
RSR: 0.5649
PBIAS: -5.3593


In [63]:
plot_predictions_plotly(y_true_pune, y_pred_pune, title="Pune's Prediction vs Ground Truth (Test)")

# Storing the model for future use

In [64]:
os.makedirs('../Models/ANN', exist_ok=True)

In [65]:
torch.save(blr_model.state_dict(), "../Models/ANN/blr_ann_model.pth")

In [66]:
torch.save(hyd_model.state_dict(), "../Models/ANN/hyd_ann_model.pth")

In [67]:
torch.save(pune_model.state_dict(), "../Models/ANN/pune_ann_model.pth")